# GameTheory-03d — Deux espèces de flèches : quand une transformation est-elle aussi un morphisme ?
**Navigation** : [<< 3b-Chambres-Murs-Chemins](GameTheory-03b-Chambres-Murs-Chemins-Lean-Python.ipynb) | [Index](README.md) | [Version C#](GameTheory-03d-Deux-Especes-de-Fleches-CSharp.ipynb) | [Version Python](GameTheory-03d-Deux-Especes-de-Fleches-Python.ipynb) | [3e-Meta-Actions-Tarifees >>](GameTheory-03e-Meta-Actions-Tarifees-Python.ipynb)


**Objectifs de ce notebook :**

1. **Distinguer deux espèces de flèches** qui coexistent sur l'espace des jeux 2x2 ordinaux : les flèches qui *transforment* la structure (les swaps de rangs de Robinson-Goforth) et les flèches qui la *préservent* (les morphismes de jeux au sens Tohmé-Viglizzo).
2. **Rendre la question finie et décidable** : sur les 576 jeux stricts et les 6 générateurs adjacents, la question « ce générateur est-il un morphisme pour ce jeu ? » est une énumération de 3456 paires — pas un slogan.
3. **Exhiber un contre-exemple concret** où un swap détruit un équilibre de Nash qu'un morphisme aurait préservé.
4. **Énumérer l'intersection** : combien de paires (jeu, générateur) le générateur préserve-t-il ? Le compte doit être reproductible.
5. **Caractériser** la préservation par une condition simple sur le jeu de départ — et la vérifier exhaustivement, pas seulement la conjecturer.

Ce notebook est la première expérience de la « Loi III — les deux espèces de flèches » sur un univers fini où tout se vérifie. Il prolonge [`GameTheory-03-Topology2x2-Python.ipynb`](GameTheory-03-Topology2x2-Python.ipynb), dont il réutilise la représentation ordinale.

**Prérequis** : jeux 2×2 et équilibres purs. **Durée** : 2 h.

## 1. Le problème : deux espèces de flèches sur le même espace

La théorie des jeux dispose de deux traditions qui font circuler des flèches entre les jeux, et **elles ne font pas la même chose** :

| Espèce | Ce qu'elle fait | Source |
|---|---|---|
| **Flèches qui PRÉSERVENT la structure** | morphismes de jeux : ils transportent les équilibres d'un jeu vers un autre jeu qui « joue pareil » | catégories concrètes de jeux de Tohmé-Viglizzo (`Gam_I`, `Gam`) |
| **Flèches qui TRANSFORMENT la structure** | swaps de rangs : ils changent les préférences d'un joueur, donc le jeu lui-même | topologie des jeux 2x2 de Robinson-Goforth |

Un **swap de rangs** échange deux valeurs dans la table de gains d'un joueur — par exemple, permuter les rangs 1 et 2 du joueur Ligne. Ce n'est pas un changement de croyances ni d'actions : c'est un **changement des préférences elles-mêmes**, donc du monde descriptif. Un **morphisme de jeux**, à l'inverse, garantit que ce qui est rationnel ici l'est encore là-bas : les équilibres se transportent.

La **Loi III** du chantier 4 s'énonce alors précisément :

> Un swap change les préférences — il n'est donc **pas automatiquement** un morphisme. La question n'est pas « transformation ou morphisme ? » en général, mais **« pour ce jeu précis, ce générateur précis, préserve-t-il les équilibres ? »**.

Et cette question, sur un univers fini, a une réponse mécanique.

### La question rendue finie

L'univers de `GameTheory-3` est l'espace des **jeux 2x2 stricts ordinaux** : chaque joueur distribue les rangs 1-4 sur les quatre cases. Il y a $24 \times 24 = 576$ tels jeux. La grammaire de Robinson-Goforth les fait bouger par **six générateurs adjacents** :

- $R_{12}, R_{23}, R_{34}$ : échanger les rangs adjacents $(1,2)$, $(2,3)$ ou $(3,4)$ dans les gains du joueur Ligne ;
- $C_{12}, C_{23}, C_{34}$ : la même chose pour le joueur Colonne.

Ces six swaps suffisent à engendrer tout le groupe $S_4 \times S_4$ des permutations de rangs (les transpositions adjacentes engendrent $S_4$) : un jeu ne se décrit pas, **il se rejoint** depuis un autre.

Sur cet espace, la question « transformation ou morphisme ? » devient une propriété **décidable par énumération** :

$$576 \text{ jeux} \times 6 \text{ générateurs} = 3456 \text{ paires à trancher}.$$

Réduction honnête de la notion de morphisme : dans les catégories de Tohmé-Viglizzo, un morphisme de jeux transporte aussi les stratégies. Ici, un swap **ne déplace aucune stratégie** — seule l'étiquette des gains change — donc l'application induite sur les profils est l'identité. L'espèce préservante se teste alors opérationnellement :

> **Définition (finie).** Le générateur $g$ est un *morphisme pour le jeu $G$* si $g$ préserve l'ensemble des équilibres de Nash purs : $\text{Nash}(G) = \text{Nash}(g(G))$, la comparaison se faisant cellule à cellule (application identité).

C'est cette définition que tout le notebook met à l'épreuve.

In [2]:
// Jeu 2x2 a representation ordinale (convention GameTheory-3).
// `Game` = { gains Ligne [4], gains Colonne [4], nom }. Table par cellule 0|1 / 2|3.
// `struct` (pas un alias de 3-tuple) : le re-parse cumule d'evcxr casse sur un type
// 3+-tuple dans une signature de fn, mais accepte un struct nominal (cf 04h Rationnel).
#[derive(Clone, PartialEq)]
struct Game { row: [i64; 4], col: [i64; 4], name: String }

fn g_new(row: [i64; 4], col: [i64; 4], name: &str) -> Game {
    let mut r = row;
    r.sort();
    assert_eq!(r, [1, 2, 3, 4], "row_payoffs doit etre une permutation de 1-4");
    let mut c = col;
    c.sort();
    assert_eq!(c, [1, 2, 3, 4], "col_payoffs doit etre une permutation de 1-4");
    Game { row, col, name: name.to_string() }
}

fn g_to_mat(g: &Game) -> ([[i64; 2]; 2], [[i64; 2]; 2]) {
    let a = [[g.row[0], g.row[1]], [g.row[2], g.row[3]]];
    let b = [[g.col[0], g.col[1]], [g.col[2], g.col[3]]];
    (a, b)
}

fn g_display(g: &Game) {
    let (a, b) = g_to_mat(g);
    let title = if g.name.is_empty() { "Jeu" } else { &g.name };
    println!("\n{}", title);
    println!("{}", "=".repeat(32));
    println!("        C0        C1");
    println!("R0  ({},{})     ({},{})", a[0][0], b[0][0], a[0][1], b[0][1]);
    println!("R1  ({},{})     ({},{})", a[1][0], b[1][0], a[1][1], b[1][1]);
}

// Encodages ordinaux des jeux classiques (identiques a GameTheory-3).
// `Vec` ordonne : la cle differe du nom affiche (la cle `Chicken` -> `Chicken (Hawk-Dove)`).
fn classic() -> Vec<(&'static str, Game)> {
    vec![
        ("Prisoner's Dilemma", g_new([3, 1, 4, 2], [3, 4, 1, 2], "Prisoner's Dilemma")),
        ("Stag Hunt", g_new([4, 1, 3, 2], [4, 3, 1, 2], "Stag Hunt")),
        ("Chicken", g_new([3, 2, 4, 1], [3, 4, 2, 1], "Chicken (Hawk-Dove)")),
        ("Matching Pennies", g_new([4, 1, 2, 3], [1, 4, 3, 2], "Matching Pennies")),
        ("Deadlock", g_new([2, 1, 4, 3], [2, 4, 1, 3], "Deadlock")),
        ("Harmony", g_new([4, 3, 2, 1], [4, 2, 3, 1], "Harmony")),
    ]
}

let _ok_3 = 0;


In [3]:

// Ligne d'environnement : le carnet Rust indique son noyau, pas la version Python.
println!("Noyau : Rust (evcxr), std seul");
for (_, g) in classic() {
    g_display(&g);
};
// point-virgule requis sous evcxr : sinon la boucle est l'expression finale du
// bloc et evcxr publie un `execute_result` `()` que la source n'a pas.


Noyau : Rust (evcxr), std seul


Prisoner's Dilemma


        C0        C1


R0  (3,3)     (1,4)


R1  (4,1)     (2,2)


Stag Hunt


... (15 outputs left out)

Harmony


        C0        C1


R0  (4,4)     (3,2)


R1  (2,3)     (1,1)


### Lecture de la représentation

Chaque case affiche `(gain Ligne, gain Colonne)` en rangs ordinaux — **le rang 4 est le meilleur** pour le joueur qui le reçoit. Deux observations qui serviront tout le long :

- **Lecture par colonne** : dans la colonne $C_j$, le joueur Ligne compare ses deux gains ; le plus grand désigne sa **meilleure réponse** dans cette colonne. Symétriquement, le joueur Colonne se lit **par ligne**.
- **Stricteté** : chaque joueur place exactement une fois chaque rang 1-4 ; il n'y a **aucune égalité**, donc chaque colonne (resp. ligne) a une meilleure réponse unique de chaque côté. C'est ce qui rend l'espace fini et propre — les jeux à égalité (les « murs » du grain D2, issue #12213) sont un autre objet.

Le Dilemme du Prisonnier se relit ainsi : pour Ligne, la colonne $C_0$ compare 3 et 4 (il préfère $R_1$ : déserter), la colonne $C_1$ compare 1 et 2 (il préfère encore $R_1$) — $R_1$ est une stratégie dominante, et l'équilibre unique $(R_1, C_1)$ est le pire outcome collectif.

In [4]:

// Equilibres de Nash purs : cases ou chaque strategie est meilleure reponse a l'autre.
fn nash(g: &Game) -> Vec<(usize, usize)> {
    let (a, b) = g_to_mat(g);
    let mut eq = Vec::new();
    for i in 0..2 {
        for j in 0..2 {
            if a[i][j] >= a[1 - i][j] && b[i][j] >= b[i][1 - j] {
                eq.push((i, j));
            }
        }
    }
    eq
}

fn nash_str(v: &[(usize, usize)]) -> String {
    if v.is_empty() {
        "aucun".to_string()
    } else {
        v.iter().map(|&(i, j)| format!("(R{},C{})", i, j))
            .collect::<Vec<String>>().join(", ")
    }
}

println!("Structure de Nash des jeux classiques :");
println!("{}", "=".repeat(52));
for (name, g) in classic() {
    let eq = nash(&g);
    println!("  {:<22} {} equilibre(s) : {}", name, eq.len(), nash_str(&eq));
};
// point-virgule requis sous evcxr : sinon la boucle est l'expression finale du
// bloc et evcxr publie un `execute_result` `()` que la source n'a pas.


Structure de Nash des jeux classiques :


  Prisoner's Dilemma     1 equilibre(s) : (R1,C1)


  Stag Hunt              2 equilibre(s) : (R0,C0), (R1,C1)


  Chicken                2 equilibre(s) : (R0,C1), (R1,C0)


  Matching Pennies       0 equilibre(s) : aucun


  Deadlock               1 equilibre(s) : (R1,C1)


  Harmony                1 equilibre(s) : (R0,C0)


### Interprétation : la structure de Nash, signature du jeu

L'ensemble des équilibres de Nash purs est la **signature stratégique** d'un jeu — c'est précisément ce qu'un morphisme de jeux s'engage à transporter. La collection classique couvre les quatre situations : un unique équilibre à dominance (Dilemme, Deadlock, Harmony), deux équilibres de coordination ou d'anti-coordination (Stag Hunt, Chicken), et **aucun équilibre pur** (Matching Pennies, où l'instabilité est le jeu).

Ce qui va nous intéresser n'est pas la classification elle-même (c'est le sujet de GameTheory-3, §5-6) mais une question plus fine : **cette signature survit-elle au passage d'un générateur ?** Un jeu sans équilibre peut en acquérir un, un jeu à équilibre unique peut le voir déménager — ou tout perdre. Chacun de ces événements est une **destruction** au sens morphique : la flèche existe (le swap est toujours applicable), mais elle ne préserve pas.

In [5]:

// Echange deux valeurs de rang dans une table de gains (convention GameTheory-3).
fn swap(payoffs: [i64; 4], r1: i64, r2: i64) -> [i64; 4] {
    let mut result = payoffs;
    for i in 0..4 {
        if result[i] == r1 {
            result[i] = r2;
        } else if result[i] == r2 {
            result[i] = r1;
        }
    }
    result
}

// Applique un des six generateurs adjacents R12/R23/R34/C12/C23/C34 (rend un jeu sans nom).
fn apply(g: &Game, gop: &str) -> Game {
    let by = gop.as_bytes();
    let r1 = (by[1] - b'0') as i64;
    let r2 = (by[2] - b'0') as i64;
    if by[0] == b'R' {
        g_new(swap(g.row, r1, r2), g.col, "")
    } else {
        g_new(g.row, swap(g.col, r1, r2), "")
    }
}

fn gens() -> [&'static str; 6] {
    ["R12", "R23", "R34", "C12", "C23", "C34"]
}

let _ok_7 = 0;


In [6]:

// Demonstration : le Dilemme du Prisonnier perd son equilibre quand on echange ses rangs 1 et 2
// cote Ligne. L'equilibre n'est pas DETRUIT (il reste un equilibre) : il SE DEPLACE.
let pd = classic().into_iter().find(|e| e.0 == "Prisoner's Dilemma").unwrap().1;
let pd_r12 = apply(&pd, "R12");

println!("Jeu de depart :");
g_display(&pd);
println!("  Nash : {}", nash_str(&nash(&pd)));

println!("\nApres le generateur R12 (rangs 1 et 2 echanges pour Ligne) :");
g_display(&pd_r12);
println!("  Nash : {}", nash_str(&nash(&pd_r12)));


Jeu de depart :


Prisoner's Dilemma


        C0        C1


R0  (3,3)     (1,4)


R1  (4,1)     (2,2)


  Nash : (R1,C1)


... (2 outputs left out)

        C0        C1


R0  (3,3)     (2,4)


R1  (4,1)     (1,2)


  Nash : (R0,C1)


### Interprétation : la grammaire en action — et le soupçon

Le générateur $R_{12}$ a échangé les rangs 1 et 2 du joueur Ligne : la case $(R_0, C_1)$ est passée de 1 à 2 et la case $(R_1, C_1)$ de 2 à 1. Les **stratégies n'ont pas bougé**, les préférences ont changé — et la sortie mesure déjà le soupçon fondateur de la Loi III : dans la colonne $C_1$, Ligne comparait 1 et 2 (il préférait $R_1$) ; il compare maintenant 2 et 1 (il préfère $R_0$). **Sa meilleure réponse dans cette colonne s'est retournée**, et avec elle l'équilibre.

Ce seul exemple montre les deux espèces à l'œuvre : la flèche $R_{12}$ **existe** toujours (c'est une transformation valide de l'espace) mais, pour ce jeu précis, elle **ne préserve pas** la signature de Nash. Reste à savoir si c'est un accident du Dilemme ou une structure de l'espace entier — c'est l'objet des sections suivantes. La réponse tiendra en une condition d'une ligne sur la table de gains du jeu de départ.

## 2. L'espèce préservante : tester le morphisme, flèche par flèche

La définition finie de la section 1 se implémente en une fonction : $g$ est un morphisme pour $G$ si et seulement si les ensembles de Nash coïncident **avant et après** l'application de $g$. Notons ce que cette comparaison ne demande pas : elle ne demande pas que le jeu soit « pareil » (les gains diffèrent en général), ni que les équilibres soient aux mêmes cases pour toujours — elle demande l'égalité des deux ensembles pour **cette** paire $(G, g)$.

C'est bien une relation *locale* : le même générateur peut être morphisme pour un jeu et pas pour son voisin. L'espace des flèches se partitionne donc en

- **flèches préservantes** — celles qui, pour ce jeu, transportent la signature de Nash intacte (l'identité sur les profils est alors compatible avec la structure de jeu) ;
- **flèches transformatrices** — celles qui cassent, déplacent ou créent des équilibres.

Et parce que l'univers est fini, chacune des 3456 paires tombe d'un côté ou de l'autre **sans zone grise**.

In [7]:

// Le generateur gop preserve-t-il l'ensemble des equilibres de Nash purs de g ?
fn morphism(g: &Game, gop: &str) -> bool {
    let before: std::collections::BTreeSet<(usize, usize)> =
        std::collections::BTreeSet::from_iter(nash(g));
    let after: std::collections::BTreeSet<(usize, usize)> =
        std::collections::BTreeSet::from_iter(nash(&apply(g, gop)));
    before == after
}

println!("Trancher quelques paires a la main :");
println!("{}", "=".repeat(56));
let paires: [(&str, &str); 6] = [
    ("Deadlock", "R12"),
    ("Deadlock", "R34"),
    ("Prisoner's Dilemma", "R12"),
    ("Prisoner's Dilemma", "C12"),
    ("Harmony", "C34"),
    ("Matching Pennies", "R23"),
];
for (game_name, gop) in paires {
    let g = classic().into_iter().find(|e| e.0 == game_name).unwrap().1;
    let g2 = apply(&g, gop);
    let verdict = morphism(&g, gop);
    let label = if verdict { "PRESERVE" } else { "TRANSFORME" };
    println!("  {:<22} + {} : Nash {:<12} -> {:<12} => {}",
        game_name, gop, nash_str(&nash(&g)), nash_str(&nash(&g2)), label);
};
// point-virgule requis sous evcxr : sinon la boucle est l'expression finale du
// bloc et evcxr publie un `execute_result` `()` que la source n'a pas.


Trancher quelques paires a la main :


  Deadlock               + R12 : Nash (R1,C1)      -> (R1,C1)      => PRESERVE


  Deadlock               + R34 : Nash (R1,C1)      -> (R1,C1)      => PRESERVE


  Prisoner's Dilemma     + R12 : Nash (R1,C1)      -> (R0,C1)      => TRANSFORME


  Prisoner's Dilemma     + C12 : Nash (R1,C1)      -> (R1,C0)      => TRANSFORME


  Harmony                + C34 : Nash (R0,C0)      -> (R0,C0)      => PRESERVE


  Matching Pennies       + R23 : Nash aucun        -> aucun        => PRESERVE


### Interprétation : deux espèces, même espace

Les six verdicts mesurés racontent déjà la Loi III :

- **Deadlock + $R_{12}$ : PRÉSERVÉ.** Pour Ligne, les colonnes comparent (2,4) et (1,3) — échanger 1 et 2 ne change aucun ordre interne de colonne : les meilleures réponses ne bougent pas, l'équilibre $(R_1, C_1)$ reste seul. La flèche transforme la *table* mais transporte la *structure* : elle est des deux espèces à la fois.
- **Prisoner's Dilemma + $R_{12}$ : TRANSFORMÉ.** La colonne $C_1$ comparait (1,2) ; après le swap elle compare (2,1) — la meilleure réponse de Ligne s'y retourne, et l'équilibre unique déménage de $(R_1, C_1)$ vers $(R_0, C_1)$.
- **Matching Pennies + $R_{23}$ : PRÉSERVÉ… trivialement.** L'ensemble vide est égal à lui-même : le jeu n'a pas d'équilibre pur avant ni après. La préservation ici ne dit rien d'intéressant — un cas limite qu'il faudra garder en tête en lisant les comptes de la section 4.

Le point structurant : **le verdict n'appartient ni au générateur ni au jeu seul, mais à la paire**. C'est exactement pourquoi la question « un swap est-il un morphisme ? » est mal posée sans son jeu — et pourquoi l'énumération qui suit n'est pas un exercice de style.

## 3. Exercice 1 : implémenter les deux espèces et produire un contre-exemple

À vous de jouer : la fonction `contre_exemple` doit trouver, dans les jeux classiques, **une paire (jeu, générateur) où le swap détruit un équilibre** — c'est-à-dire où un équilibre de Nash du jeu de départ n'est plus équilibre dans le jeu d'arrivée. Un contre-exemple, pas un principe.

In [8]:

// (exercice) La reponse attendue est une structure { jeu, generateur, nash avant, nash apres, detruit }.
// `struct` (pas un alias de 5-tuple) : le re-parse cumule d'evcxr casse sur un type 3+-tuple dans une signature.
struct Rep1 { jeu: String, gop: String, avant: Vec<(usize, usize)>, apres: Vec<(usize, usize)>, detruit: Vec<(usize, usize)> }

fn contre_exemple(games: &[(&str, Game)]) -> Option<Vec<Rep1>> {
    // TODO etudiant : trouver les paires (jeu, generateur) ou le swap DETRUIT un equilibre
    // Etape 1 : pour chaque jeu et chaque generateur des 6, calculer Nash avant et Nash apres
    // Etape 2 : garder les paires ou un equilibre du depart disparait (destruction stricte :
    //           un profil de Nash(G) absent de Nash(g(G)) -- pas seulement un deplacement)
    // Etape 3 : retourner une liste de {jeu, generateur, nash_avant, nash_apres, detruit}
    // Indice : morphism dit si les ENSEMBLES sont egaux ; ici il faut regarder
    //          profil par profil -- un deplacement (ensemble different) n'est pas une destruction
    // Verifiez votre resultat : au moins un classique doit apparaitre avec une destruction stricte
    let _ = games;
    None // TODO etudiant : remplacer
}

println!("Exercice a completer : contre-exemple de destruction d'equilibre par un swap");


Exercice a completer : contre-exemple de destruction d'equilibre par un swap


### Vérification de l’exercice 1
Un témoin doit perdre un profil sans en créer un nouveau ; une liste vide ne suffit pas.


In [9]:

fn py_bool(b: bool) -> String {
    if b { "True".to_string() } else { "False".to_string() }
}

fn valider_contre_exemple(reponses: &Option<Vec<Rep1>>,
                          jeux: &[(&str, Game)]) -> bool {
    use std::collections::BTreeSet;
    let inner = match reponses {
        Some(v) => v,
        None => return false,
    };
    if inner.is_empty() {
        return false;
    }
    for r in inner {
        let g = match jeux.iter().find(|e| e.0 == r.jeu.as_str()) {
            Some(e) => &e.1,
            None => return false,
        };
        if !gens().contains(&r.gop.as_str()) {
            return false;
        }
        let avant: BTreeSet<(usize, usize)> = BTreeSet::from_iter(nash(g));
        let apres: BTreeSet<(usize, usize)> =
            BTreeSet::from_iter(nash(&apply(g, &r.gop)));
        let r_avant: BTreeSet<(usize, usize)> = BTreeSet::from_iter(r.avant.clone());
        let r_apres: BTreeSet<(usize, usize)> = BTreeSet::from_iter(r.apres.clone());
        let r_detruit: BTreeSet<(usize, usize)> = BTreeSet::from_iter(r.detruit.clone());
        let detruit = avant.difference(&apres).copied().collect::<BTreeSet<_>>();
        let correct = r_avant == avant
            && r_apres == apres
            && r_detruit == detruit
            && !detruit.is_empty()
            && apres.difference(&avant).next().is_none();
        if !correct {
            return false;
        }
    }
    true
}

let classiques = classic();
let reponse = contre_exemple(&classiques);
if reponse.is_none() {
    println!("Exercice 1 à compléter");
} else {
    let mut contrat = valider_contre_exemple(&reponse, &classiques);
    for nom in ["Stag Hunt", "Chicken"] {
        let ju: Vec<(&str, Game)> = classiques.iter().filter(|e| e.0 == nom).cloned().collect();
        contrat = contrat && valider_contre_exemple(&contre_exemple(&ju), &ju);
    }
    println!("Contrat exercice 1 : {}", py_bool(contrat));
};
// point-virgule requis sous evcxr : sinon le bloc if/else est l'expression
// finale et evcxr publie un `execute_result` `()` que la source n'a pas.


Exercice 1 à compléter


### Exemple résolu : un équilibre détruit, un équilibre déplacé

La démonstration guidée ci-dessous tranche la question sur le Dilemme du Prisonnier avec les deux générateurs qui l'attaquent le plus naturellement : $R_{12}$ (Ligne) et $C_{12}$ (Colonne). On regarde les profils un par un — c'est la distinction destruction/déplacement qui compte :

- **déplacement** : l'équilibre de départ n'est plus là, mais un autre est apparu (la cardinalité est conservée) ;
- **destruction stricte** : un profil de Nash($G$) disparaît et aucun nouveau profil ne le compense.

In [10]:

// (exemple resolu, pas un exercice) Compare Nash avant/apres, profil par profil.
// `struct` (pas un alias de 4-tuple) : le re-parse cumule d'evcxr casse sur un type 3+-tuple dans une signature.
struct Diff {
    avant: std::collections::BTreeSet<(usize, usize)>,
    apres: std::collections::BTreeSet<(usize, usize)>,
    detruits: std::collections::BTreeSet<(usize, usize)>,
    creees: std::collections::BTreeSet<(usize, usize)>,
}

fn differencial(g: &Game, gop: &str) -> Diff {
    let avant: std::collections::BTreeSet<(usize, usize)> =
        std::collections::BTreeSet::from_iter(nash(g));
    let apres: std::collections::BTreeSet<(usize, usize)> =
        std::collections::BTreeSet::from_iter(nash(&apply(g, gop)));
    let detruits = avant.difference(&apres).copied().collect();
    let creees = apres.difference(&avant).copied().collect();
    Diff { avant, apres, detruits, creees }
}

let pd = classic().into_iter().find(|e| e.0 == "Prisoner's Dilemma").unwrap().1;
println!("Jeu : Prisoner's Dilemma   Nash initial : {}", nash_str(&nash(&pd)));
println!("{}", "=".repeat(62));
for gop in ["R12", "C12", "R34"] {
    let d = differencial(&pd, gop);
    let d_det: Vec<(usize, usize)> = d.detruits.iter().copied().collect();
    let d_cre: Vec<(usize, usize)> = d.creees.iter().copied().collect();
    let nature = if !d.detruits.is_empty() && !d.creees.is_empty() {
        format!("DEPLACEMENT : {} -> {}", nash_str(&d_det), nash_str(&d_cre))
    } else if !d.detruits.is_empty() {
        format!("DESTRUCTION stricte : {}", nash_str(&d_det))
    } else if d.creees.is_empty() {
        "preservation".to_string()
    } else {
        format!("preservation (deplacement : {})", nash_str(&d_cre))
    };
    let apres_str = nash_str(&d.apres.iter().copied().collect::<Vec<_>>());
    println!("  {} : Nash apres {:<12} | {}", gop, apres_str, nature);
}

// Le contre-exemple attendu par l'exercice 1 existe-t-il dans les classiques ?
println!();
println!("Recherche d'une DESTRUCTION stricte dans les jeux classiques :");
let mut trouve = false;
for (name, g) in classic() {
    for gop in gens() {
        let d = differencial(&g, gop);
        if !d.detruits.is_empty() && d.creees.is_empty() {
            trouve = true;
            let det: Vec<(usize, usize)> = d.detruits.iter().copied().collect();
            println!("  {} + {} : perd {} sans compensation", name, gop, nash_str(&det));
        }
    }
}
if !trouve {
    println!("  (aucune dans cet echantillon -- elargir l'espace)");
};
// point-virgule requis sous evcxr : sinon le bloc if est l'expression
// finale et evcxr publie un `execute_result` `()` que la source n'a pas.


Jeu : Prisoner's Dilemma   Nash initial : (R1,C1)


  R12 : Nash apres (R0,C1)      | DEPLACEMENT : (R1,C1) -> (R0,C1)


  C12 : Nash apres (R1,C0)      | DEPLACEMENT : (R1,C1) -> (R1,C0)


  R34 : Nash apres (R1,C1)      | preservation


Recherche d'une DESTRUCTION stricte dans les jeux classiques :


  Stag Hunt + R12 : perd (R1,C1) sans compensation


... (3 outputs left out)

  Chicken + R12 : perd (R0,C1) sans compensation


  Chicken + R34 : perd (R1,C0) sans compensation


  Chicken + C12 : perd (R1,C0) sans compensation


  Chicken + C34 : perd (R0,C1) sans compensation


### Interprétation : trois régimes mesurés sur un seul jeu — et le mur habité

Le Dilemme exhibe d'un coup les **trois régimes** possibles d'une flèche :

- **$R_{12}$ et $C_{12}$ : déplacement.** L'équilibre unique déménage ($(R_1,C_1) \to (R_0,C_1)$ pour le premier, $\to (R_1,C_0)$ pour le second). L'ensemble change — pas morphisme — mais le jeu d'arrivée a toujours un équilibre : la richesse stratégique est transportée, pas amputée.
- **$R_{34}$ : préservation.** La colonne $C_0$ du Dilemme contient bien le couple adjacent $(3,4)$ pour Ligne — la flèche **traverse un mur** — et pourtant la signature de Nash ne bouge pas. Premier indice que la condition naïve « couple adjacent ⟹ transformation » est **fausse** : il faut comprendre pourquoi ce passage est invisible.
- Sur les autres classiques, **Stag Hunt et Chicken subissent des destructions strictes** : chacun de leurs deux équilibres peut être supprimé sans compensation par le bon générateur. C'est le contre-exemple dur que l'exercice 1 demande — une coordination qui **perd** une de ses sorties.

Le vocabulaire géométrique du chantier 4 s'impose : dans la lecture Bruns-Kimmich (grain D2, issue #12213), les jeux stricts sont des **chambres** et les swaps adjacents traversent des **murs**. Mais le régime $R_{34}$ du Dilemme montre la subtilité : **un mur ne change la chambre Nash que s'il est habité** — si aucune meilleure réponse de Colonne ne pointe vers la colonne traversée, aucun équilibre ne vivait contre ce mur, et le passage est stratégiquement invisible. C'est cette intuition que la section 6 transforme en théorème vérifié.

## 4. Énumération exhaustive : l'intersection des deux espèces

L'échantillon des classiques ne suffit pas : la Loi III demande le compte **sur l'espace entier**. L'énumération ci-dessous génère les 576 jeux stricts (toutes paires de permutations de 1-4), applique chacun des 6 générateurs à chacun des jeux, et tranche les 3456 paires une par une. Trois mesures en sortent :

1. le compte **par générateur** : combien de jeux chaque générateur préserve ;
2. la **distribution par jeu** : combien de générateurs transforment chaque jeu (de 0 à 6) ;
3. les **points fixes de la grammaire** : les jeux que les six générateurs préservent — les jeux « au cœur de leur chambre », inatteignables par un seul pas de la grammaire.

In [11]:

// Les 576 jeux 2x2 stricts : 24 permutations de rangs par joueur (ordre lexicographique).
fn all_games() -> Vec<Game> {
    let mut games = Vec::new();
    for a in 1..=4i64 {
        for b in 1..=4i64 {
            for c in 1..=4i64 {
                for d in 1..=4i64 {
                    if a == b || a == c || a == d || b == c || b == d || c == d {
                        continue;
                    }
                    for a2 in 1..=4i64 {
                        for b2 in 1..=4i64 {
                            for c2 in 1..=4i64 {
                                for d2 in 1..=4i64 {
                                    if a2 == b2 || a2 == c2 || a2 == d2
                                        || b2 == c2 || b2 == d2 || c2 == d2 {
                                        continue;
                                    }
                                    games.push(g_new([a, b, c, d], [a2, b2, c2, d2], ""));
                                }
                            }
                        }
                    }
                }
            }
        }
    }
    games
}

let all = all_games();
println!("Jeux stricts generes : {}  (24 x 24)", all.len());
println!();

// 1. Compte par generateur
println!("Preservation par generateur (sur 576 jeux) :");
println!("{}", "=".repeat(46));
let mut total = 0usize;
for gop in gens() {
    let n_pres = all.iter().filter(|g| morphism(g, gop)).count();
    total += n_pres;
    println!("  {} : {:3} preserves / {:3} transformes  ({:.1} %)",
        gop, n_pres, 576 - n_pres, (100.0 * n_pres as f64) / 576.0);
}
println!("  {:>4} : {} paires preservantes sur {} ({:.1} %)",
    "TOTAL", total, 576 * 6, (100.0 * total as f64) / (576.0 * 6.0));

// 2. Distribution par jeu : combien de generateurs transforment ce jeu
println!();
println!("Distribution du nombre de generateurs transformateurs par jeu :");
let mut par_jeu: std::collections::BTreeMap<usize, usize> = std::collections::BTreeMap::new();
for g in &all {
    let n_trans = gens().iter().filter(|gop| !morphism(g, *gop)).count();
    *par_jeu.entry(n_trans).or_insert(0) += 1;
}
for (k, v) in par_jeu.iter() {
    let barre = "#".repeat(*v / 6);
    println!("  {} generateur(s) transformateur(s) : {:3} jeux  {}", k, v, barre);
}

// 3. Points fixes de la grammaire : 0 generateur transformateur
let n_fixes = all.iter()
    .filter(|g| gens().iter().all(|gop| morphism(g, *gop)))
    .count();
println!("\nPoints fixes (6/6 preserves) : {} jeux", n_fixes);


Jeux stricts generes : 576  (24 x 24)


Preservation par generateur (sur 576 jeux) :


  R12 : 432 preserves / 144 transformes  (75.0 %)


  R23 : 432 preserves / 144 transformes  (75.0 %)


  R34 : 432 preserves / 144 transformes  (75.0 %)


  C12 : 432 preserves / 144 transformes  (75.0 %)


... (6 outputs left out)

  2 generateur(s) transformateur(s) : 180 jeux  ##############################


  3 generateur(s) transformateur(s) :  80 jeux  #############


  4 generateur(s) transformateur(s) :  16 jeux  ##


Points fixes (6/6 preserves) : 100 jeux


### Interprétation : la séparation nette existe — et elle est massive

Les comptes mesurés établissent la Loi III sur l'univers entier :

- **Chaque générateur préserve 432 jeux sur 576 (75 %) et en transforme 144 (25 %).** Un quart de l'espace change de signature de Nash à chaque pas de la grammaire — la séparation n'est ni marginale ni décorative. Et aucun générateur n'est « globalement » morphisme : l'espèce d'une flèche est une propriété de la paire.
- **La distribution par jeu est unanime** : 100 jeux sont préservés par les six générateurs — les « points fixes de la grammaire », et 16 jeux sont transformés par **quatre** générateurs sur six — les plus exposés de l'espace. La masse (460 jeux) se concentre entre un et trois générateurs transformateurs.
- **2592 paires préservantes sur 3456** — dont les préservations triviales (les jeux sans équilibre pur qui n'en acquièrent pas après le swap) et les préservations *invisibles* découvertes en section 3 ($R_{34}$ sur le Dilemme) : le compte honnête les inclut, la lecture doit s'en souvenir.

Le verdict du grain D5 s'écrit : **la séparation nette apparaît** — certaines flèches préservent, d'autres transforment, et le partage n'est ni uniforme ni arbitraire. La distribution a une forme ; une forme annonce une cause. La section suivante commence par se tromper dessus — c'est voulu.

## 5. Exercice 2 : cartographier l'intersection par structure de Nash

Le compte global dit *combien* ; à vous de dire *quoi*. La fonction `cartographier_intersection` doit croiser l'énumération avec la **structure de Nash** des jeux de départ (nombre d'équilibres purs) pour répondre : les jeux préservés par un générateur sont-ils surtout des jeux sans équilibre (préservation triviale) ou surtout des jeux à équilibre unique ? La réponse mesure à quel point la préservation mesurée en section 4 est *substantielle*.

In [12]:

// (exercice) Les compteurs par strate sont une paire (preserves, transformes).
type Compteurs = (usize, usize);

fn cartographier_intersection(games: &[Game])
    -> Option<std::collections::BTreeMap<usize, Compteurs>> {
    // TODO etudiant : croiser preservation x structure de Nash des jeux de depart
    // Etape 1 : pour chaque jeu, calculer son nombre d'equilibres purs (0 a 4)
    // Etape 2 : pour chaque paire (jeu, generateur), enregistrer preserve/transforme
    // Etape 3 : aggreger en {nb_equilibres: {preserves: X, transformes: Y}}
    // Indice : nash et morphism font tout le travail ;
    //          la question interessante est le RATIO preserves/transformes par strate
    // Verifiez : un jeu sans equilibre peut en acquerir ; mesurez les deux categories.
    let _ = games;
    None // TODO etudiant : remplacer
}

println!("Exercice a completer : cartographie preservation x structure de Nash");


Exercice a completer : cartographie preservation x structure de Nash


### Vérification de l’exercice 2
La garde vérifie les clés et chaque compteur sur les jeux fournis.


In [13]:

fn valider_carte(games: &[Game],
                 result: &Option<std::collections::BTreeMap<usize, Compteurs>>) -> bool {
    use std::collections::{BTreeMap, BTreeSet};
    let map = match result {
        Some(m) => m,
        None => return false,
    };
    let keys: BTreeSet<usize> = games.iter().map(|g| nash(g).len()).collect();
    if map.keys().copied().collect::<BTreeSet<_>>() != keys {
        return false;
    }
    for (k, compteurs) in map.iter() {
        let subset: Vec<&Game> = games.iter().filter(|g| nash(g).len() == *k).collect();
        let total = subset.iter()
            .flat_map(|g| gens().iter().map(move |gop| morphism(g, *gop)).collect::<Vec<_>>())
            .filter(|&b| b).count();
        if compteurs.0 != total || compteurs.1 != 6 * subset.len() - total {
            return false;
        }
    }
    true
}

let all = all_games();
let carte = cartographier_intersection(&all);
if carte.is_none() {
    println!("Exercice 2 à compléter");
} else {
    let classiques: Vec<Game> = classic().into_iter().map(|(_, g)| g).collect();
    let contrat = valider_carte(&all, &carte)
        && valider_carte(&classiques, &cartographier_intersection(&classiques));
    println!("Contrat exercice 2 : {}", py_bool(contrat));
};
// point-virgule requis sous evcxr : sinon le bloc if/else est l'expression
// finale et evcxr publie un `execute_result` `()` que la source n'a pas.


Exercice 2 à compléter


## 6. Caractérisation : quand un générateur préserve-t-il ?

L'énumération a une forme ; cherchons la cause. Premier pas sûr — la mécanique des meilleures réponses :

1. **Un swap adjacent $R_{(a,b)}$ ne peut retourner une meilleure réponse que dans une colonne qui contient les deux rangs échangés.** Dans une colonne donnée, la meilleure réponse de Ligne ne dépend que de l'**ordre** de ses deux gains ; échanger les valeurs $a$ et $b$ ne change aucun ordre… sauf dans une colonne dont les deux gains sont précisément $a$ et $b$ (l'ordre s'y inverse).

On serait tenté de conclure : *« couple $\{a,b\}$ présent dans une colonne ⟹ transformation. »* C'est la **conjecture naïve** — et elle est **fausse** : le régime $R_{34}$ du Dilemme (section 3) la contredit déjà. La cellule suivante la teste sur tout l'espace, montre ses désaccords, puis construit la bonne condition en regardant **pourquoi** certains retournements sont invisibles :

2. **Un retournement n'est visible dans l'ensemble de Nash que si la colonne traversée est aussi une colonne-maison pour l'autre joueur.** Le profil $(i, C_j)$ n'est équilibre que si $C_j$ est *aussi* la meilleure réponse de Colonne en ligne $i$. Si les meilleures réponses de Colonne (dans les deux lignes) pointent toutes **ailleurs** que vers $C_j$, aucun équilibre ne vit dans la colonne $C_j$ — ni avant ni après le retournement de Ligne : le mur est traversé, mais personne n'habitait contre.

D'où la **caractérisation attendue pour les swaps adjacents** : $R_{(a,b)}$ transforme $G$ si et seulement si une colonne des gains de Ligne contient $\{a, b\}$ **et** cette colonne est la meilleure réponse de Colonne dans au moins une ligne (symétrique pour $C_{(a,b)}$ : la ligne contenant $\{a,b\}$ doit être la meilleure réponse de Ligne dans au moins une colonne). L'univers étant fini, tout cela se **vérifie**.

In [14]:

// Ordre d'iteration d'un `set` CPython (>= 3.8, 64 bits) dont les elements sont haches comme des
// tuples d'entiers : hachage xxHash des tuples (Objects/tupleobject.c) et table a sondage
// (Objects/setobject.c : 8 cases au depart, LINEAR_PROBES = 9, PERTURB_SHIFT = 5, agrandissement
// a used * 4 quand fill * 5 >= mask * 3). Sans suppression, la table ne contient jamais de case
// « dummy ». Le hachage d'un entier Python est l'entier lui-meme (hors -1) : aucune graine.
fn py_tuple_hash(lanes: &[i64]) -> i64 {
    const P1: u64 = 11400714785074694791;
    const P2: u64 = 14029467366897019727;
    const P5: u64 = 2870177450012600261;
    let mut acc: u64 = P5;
    for &lane in lanes {
        acc = acc.wrapping_add((lane as u64).wrapping_mul(P2));
        acc = acc.rotate_left(31);
        acc = acc.wrapping_mul(P1);
    }
    acc = acc.wrapping_add((lanes.len() as u64) ^ (P5 ^ 3527539));
    if acc == u64::MAX {
        return 1546275796;
    }
    acc as i64
}

fn py_set_insert<T: Clone + PartialEq>(table: &mut Vec<Option<(T, i64)>>, key: &T, h: i64, compare: bool) -> bool {
    let mask = table.len() - 1;
    let mut perturb = h as u64;
    let mut i = (h as u64 as usize) & mask;
    loop {
        let probes = if i + 9 <= mask { 9 } else { 0 };
        for j in i..=i + probes {
            match &table[j] {
                None => {
                    table[j] = Some((key.clone(), h));
                    return true;
                }
                Some((k, kh)) => {
                    if compare && *kh == h && k == key {
                        return false;
                    }
                }
            }
        }
        perturb >>= 5;
        i = i.wrapping_mul(5).wrapping_add(1).wrapping_add(perturb as usize) & mask;
    }
}

// `items` dans l'ordre ou la source les ajoute au set ; rend les elements distincts dans l'ordre
// ou `for x in le_set` les parcourt.
fn py_set_order<T: Clone + PartialEq>(items: &[T], hash: impl Fn(&T) -> i64) -> Vec<T> {
    let mut table: Vec<Option<(T, i64)>> = vec![None; 8];
    let (mut fill, mut used) = (0usize, 0usize);
    for key in items {
        let h = hash(key);
        if py_set_insert(&mut table, key, h, true) {
            fill += 1;
            used += 1;
            if fill * 5 >= (table.len() - 1) * 3 {
                let minused = if used > 50000 { used * 2 } else { used * 4 };
                let mut newsize = 8;
                while newsize <= minused {
                    newsize <<= 1;
                }
                let old = std::mem::replace(&mut table, vec![None; newsize]);
                for (k, kh) in old.into_iter().flatten() {
                    py_set_insert(&mut table, &k, kh, false);
                }
            }
        }
    }
    table.into_iter().flatten().map(|(k, _)| k).collect()
}

fn fmt_tuple4(t: &[i64; 4]) -> String {
    format!("({}, {}, {}, {})", t[0], t[1], t[2], t[3])
}

fn fmt_profile_list(p: &[(usize, usize)]) -> String {
    if p.is_empty() {
        "[]".to_string()
    } else {
        let parts: Vec<String> = p.iter().map(|&(i, j)| format!("({}, {})", i, j)).collect();
        format!("[{}]", parts.join(", "))
    }
}

// Colonne (0 ou 1) dont les gains Ligne contiennent {r1, r2}, sinon None.
fn colonne_contenant(g: &Game, r1: i64, r2: i64) -> Option<usize> {
    let p0 = [g.row[0], g.row[2]];
    let p1 = [g.row[1], g.row[3]];
    if (p0[0] == r1 && p0[1] == r2) || (p0[0] == r2 && p0[1] == r1) {
        return Some(0);
    }
    if (p1[0] == r1 && p1[1] == r2) || (p1[0] == r2 && p1[1] == r1) {
        return Some(1);
    }
    None
}

struct GenParts { player: char, r1: i64, r2: i64 }

fn gen_parts(gop: &str) -> GenParts {
    let by = gop.as_bytes();
    GenParts { player: by[0] as char, r1: (by[1] - b'0') as i64, r2: (by[2] - b'0') as i64 }
}

// Conjecture naive : preserve ssi le joueur vise n'a pas {a,b} dans une colonne/ligne.
fn condition_naive(g: &Game, gop: &str) -> bool {
    let gp = gen_parts(gop);
    let (player, r1, r2) = (gp.player, gp.r1, gp.r2);
    if player == 'R' {
        return colonne_contenant(g, r1, r2).is_none();
    }
    let row0 = [g.col[0], g.col[1]];
    let row1 = [g.col[2], g.col[3]];
    if (row0[0] == r1 && row0[1] == r2) || (row0[0] == r2 && row0[1] == r1) {
        return false;
    }
    if (row1[0] == r1 && row1[1] == r2) || (row1[0] == r2 && row1[1] == r1) {
        return false;
    }
    true
}

// Meilleure reponse de Colonne dans chaque ligne : [beta(0), beta(1)].
fn br_colonne(g: &Game) -> [usize; 2] {
    let (_, b) = g_to_mat(g);
    [if b[0][0] > b[0][1] { 0 } else { 1 },
     if b[1][0] > b[1][1] { 0 } else { 1 }]
}

// Meilleure reponse de Ligne dans chaque colonne : [rho(0), rho(1)].
fn br_ligne(g: &Game) -> [usize; 2] {
    let (a, _) = g_to_mat(g);
    [if a[0][0] > a[1][0] { 0 } else { 1 },
     if a[0][1] > a[1][1] { 0 } else { 1 }]
}

// Caracterisation etablie : gop preserve g ssi le joueur vise n'a PAS {a,b} dans
// une colonne/ligne, OU cette colonne/ligne n'est la meilleure reponse de l'AUTRE
// joueur dans AUCUNE ligne/colonne (le mur traverse n'est habite par personne).
fn condition_verifiee(g: &Game, gop: &str) -> bool {
    let gp = gen_parts(gop);
    let (player, r1, r2) = (gp.player, gp.r1, gp.r2);
    if player == 'R' {
        let j = colonne_contenant(g, r1, r2);
        let br = br_colonne(g);
        return match j {
            Some(j) => !br.contains(&j),
            None => true,
        };
    }
    let row0 = [g.col[0], g.col[1]];
    let row1 = [g.col[2], g.col[3]];
    let i = if (row0[0] == r1 && row0[1] == r2) || (row0[0] == r2 && row0[1] == r1) {
        Some(0)
    } else if (row1[0] == r1 && row1[1] == r2) || (row1[0] == r2 && row1[1] == r1) {
        Some(1)
    } else {
        None
    };
    let br = br_ligne(g);
    match i {
        Some(i) => !br.contains(&i),
        None => true,
    }
}

let _ok_27 = 0;


In [15]:

let all = py_set_order(&all_games(),
    |g| py_tuple_hash(&[py_tuple_hash(&g.row), py_tuple_hash(&g.col)]));

// Etape 1 : la conjecture naive echoue -- mesurons son echec
let mut desaccords_naif: Vec<(Game, &str)> = Vec::new();
for g in &all {
    for gop in gens() {
        if morphism(g, gop) != condition_naive(g, gop) {
            desaccords_naif.push((g.clone(), gop));
        }
    }
}
println!("Conjecture naive testee sur {} paires : {} desaccords",
    576 * 6, desaccords_naif.len());
let g_ex = desaccords_naif[0].0.clone();
let gen_ex = desaccords_naif[0].1;
println!("  contre-exemple : R={} C={} + {}",
    fmt_tuple4(&g_ex.row), fmt_tuple4(&g_ex.col), gen_ex);
let nash_avant = fmt_profile_list(
    &std::collections::BTreeSet::from_iter(nash(&g_ex)).into_iter().collect::<Vec<_>>());
let nash_apres = fmt_profile_list(
    &std::collections::BTreeSet::from_iter(nash(&apply(&g_ex, gen_ex)))
        .into_iter().collect::<Vec<_>>());
println!("    Nash avant {} / apres {} -> mur traverse mais equilibres intacts",
    nash_avant, nash_apres);

// Etape 2 : la condition verifiee predit-elle exactement les verdicts ?
let (mut accords, mut desaccords) = (0i64, 0i64);
for g in &all {
    for gop in gens() {
        if morphism(g, gop) == condition_verifiee(g, gop) {
            accords += 1;
        } else {
            desaccords += 1;
            println!("  DESACCORD : {}/{}/ + {}",
                fmt_tuple4(&g.row), fmt_tuple4(&g.col), gop);
        }
    }
}
println!("\nCondition verifiee sur {} paires : accords = {}, desaccords = {}",
    576 * 6, accords, desaccords);
assert!(desaccords == 0, "la caracterisation echoue sur au moins une paire");
println!("La condition predit exactement les 3456 verdicts : theoreme etabli.");

// Etape 3 : le compte se DERIVE au lieu d'etre constate
let mut sans_paire: Vec<[i64; 4]> = Vec::new();
for a in 1..=4i64 {
    for b in 1..=4i64 {
        for c in 1..=4i64 {
            for d in 1..=4i64 {
                if a == b || a == c || a == d || b == c || b == d || c == d {
                    continue;
                }
                let p = [a, b, c, d];
                let c0 = [p[0], p[2]];
                let c1 = [p[1], p[3]];
                let pair_c0 = (c0[0] == 1 && c0[1] == 2) || (c0[0] == 2 && c0[1] == 1);
                let pair_c1 = (c1[0] == 1 && c1[1] == 2) || (c1[0] == 2 && c1[1] == 1);
                if !pair_c0 && !pair_c1 {
                    sans_paire.push(p);
                }
            }
        }
    }
}
println!("\nDerivation pour R12 : {} permutations sans paire (sur 24)", sans_paire.len());
println!("  -> {} x 24 = {} jeux preserves sans mur",
    sans_paire.len(), sans_paire.len() * 24);
println!("  + 8 permutations avec paire x 6 arrangements Colonne ou le mur est inhabite");
println!("  = {} + {} = {} jeux preserves attendus",
    sans_paire.len() * 24, 8 * 6, sans_paire.len() * 24 + 8 * 6);
let mesure = all.iter().filter(|g| morphism(g, "R12")).count();
println!("  mesure section 4 : {}", mesure);


Conjecture naive testee sur 3456 paires : 288 desaccords


  contre-exemple : R=(1, 3, 2, 4) C=(2, 4, 1, 3) + R12


    Nash avant [(1, 1)] / apres [(1, 1)] -> mur traverse mais equilibres intacts


Condition verifiee sur 3456 paires : accords = 3456, desaccords = 0


La condition predit exactement les 3456 verdicts : theoreme etabli.


Derivation pour R12 : 16 permutations sans paire (sur 24)


  -> 16 x 24 = 384 jeux preserves sans mur


  + 8 permutations avec paire x 6 arrangements Colonne ou le mur est inhabite


  = 384 + 48 = 432 jeux preserves attendus


  mesure section 4 : 432


### Interprétation : le théorème vérifié — verdict explicite du grain D5

La scène s'est jouée en deux actes, et c'est elle-même un enseignement :

- **Acte 1 — la conjecture naïve meurt mesurée.** « Couple adjacent en colonne ⟹ transformation » échoue sur 288 paires : tous les cas de **murs inhabités**. Sur un univers fini, une conjecture ne se discute pas, elle se compte.
- **Acte 2 — la condition pour un swap adjacent prédit les 3456 verdicts sans exception.** $R_{(a,b)}$ transforme $G$ ssi la colonne qui contient $\{a,b\}$ dans les gains de Ligne est **aussi** une colonne de meilleure réponse de Colonne. Et le compte s'en **dérive** : $16 \times 24$ jeux sans mur + $8 \times 6$ jeux au mur inhabité $= 384 + 48 = 432$ — exactement la mesure de la section 4. Le compte n'est plus un constat, c'est une conséquence.

**Verdict de l'exercice 3 de l'issue (explicité) :** oui, il existe une condition simple sur le jeu de départ qui prédit la préservation. Elle n'est pas *purement* une condition sur le joueur visé (comme le suggérait l'intuition) : elle **couple les deux tables** — la géométrie des rangs d'un côté, la direction des meilleures réponses de l'autre. C'est plus riche que prévu, et c'est vérifié exhaustivement.

Relue en langage géométrique : **traverser un mur ne change de chambre Nash que si le mur est habité** — si aucune meilleure réponse de l'autre joueur ne pointe vers la colonne traversée, aucun équilibre ne vivait contre ce mur et le passage est stratégiquement invisible. La grammaire (swaps de rangs) et la géométrie (chambres/murs du grain D2) racontent la même histoire : les transitions se jouent *là où les préférences des deux joueurs se rencontrent*.

## 7. Exercice 3 : la condition survit-elle aux swaps non adjacents ?

Les six générateurs adjacents engendrent tout $S_4 \times S_4$, mais la caractérisation de la section 6 porte sur un **seul swap adjacent**. Pour les swaps non adjacents $(1,3)$, $(1,4)$ et $(2,4)$, mesurez la préservation sur les 576 jeux et comparez-la à la condition des colonnes de la section 6. Un rang intermédiaire peut inverser l'ordre d'une colonne qui ne contient pas les deux rangs échangés : la généralisation demande donc une vérification, et non une conclusion anticipée.

In [16]:

// (exercice) Stub.
fn preservations_swap_general(g: &Game, player: char, a: i64, b: i64) -> Option<bool> {
    // TODO etudiant : generaliser la preservation aux swaps NON adjacents
    // Etape 1 : ecrire le swap (a, b) quelconque cote player (R ou C)
    // Etape 2 : mesurer la preservation comme en section 2
    // Etape 3 : comparer les mesures a la condition de la section 6, sans la supposer vraie
    // Indice : swap accepte deja n'importe quelle paire de rangs ;
    //          testez au minimum (1,3), (1,4), (2,4) -- les trois paires non adjacentes
    // Question bonus : combien de jeux chaque swap non adjacent preserve-t-il ?
    let _ = (g, player, a, b);
    None // TODO etudiant : remplacer
}

println!("Exercice a completer : generalisation aux swaps non adjacents");


Exercice a completer : generalisation aux swaps non adjacents


### Vérification de l’exercice 3
La garde confronte la réponse aux six familles de swaps non adjacents et rejette les réponses constantes.


In [17]:

fn valider_swap_general(all: &[Game],
                         f: fn(&Game, char, i64, i64) -> Option<bool>) -> bool {
    for jeu in all {
        for joueur in ['R', 'C'] {
            for (a, b) in [(1i64, 3i64), (1i64, 4i64), (2i64, 4i64)] {
                let valeur = f(jeu, joueur, a, b);
                let attendu = morphism(jeu, &format!("{}{}{}", joueur, a, b));
                if valeur != Some(attendu) {
                    return false;
                }
            }
        }
    }
    true
}

let all = all_games();
let stag = classic().into_iter().find(|e| e.0 == "Stag Hunt").unwrap().1;
if preservations_swap_general(&stag, 'R', 1, 3).is_none() {
    println!("Exercice 3 à compléter");
} else {
    println!("Contrat exercice 3 : {}",
        py_bool(valider_swap_general(&all, preservations_swap_general)));
};
// point-virgule requis sous evcxr : sinon le bloc if/else est l'expression
// finale et evcxr publie un `execute_result` `()` que la source n'a pas.


Exercice 3 à compléter


### Contre-exemple mesuré pour un swap non adjacent
La comparaison ci-dessous exhibe une meilleure réponse retournée sans colonne contenant les deux rangs échangés. Elle ne résout pas le décompte demandé dans l’exercice.


In [18]:

let all = py_set_order(&all_games(),
    |g| py_tuple_hash(&[py_tuple_hash(&g.row), py_tuple_hash(&g.col)]));

// Temoin en 3 bindings separes (pas un 3-tuple) : le re-parse cumule d'evcxr
// casse sur un type 3+-tuple meme hors signature de fn.
let mut temoin_src: usize = 0;
let mut temoin_col: usize = 0;
let mut temoin: Option<Game> = None;
for (idx, jeu) in all.iter().enumerate() {
    let trans = apply(jeu, "R13");
    for j in 0..2 {
        let colpair = [jeu.row[j], jeu.row[2 + j]];
        let has_13 = (colpair[0] == 1 && colpair[1] == 3)
            || (colpair[0] == 3 && colpair[1] == 1);
        if !has_13 && br_ligne(jeu)[j] != br_ligne(&trans)[j] {
            temoin = Some(trans);
            temoin_src = idx;
            temoin_col = j;
            break;
        }
    }
    if temoin.is_some() {
        break;
    }
}
let trans = temoin.unwrap();
let jeu = all[temoin_src].clone();
let colonne = temoin_col;
println!("R13 : gains Ligne {} -> {} ; colonne {}",
    fmt_tuple4(&jeu.row), fmt_tuple4(&trans.row), colonne);
println!("meilleure réponse : {} -> {}",
    br_ligne(&jeu)[colonne], br_ligne(&trans)[colonne]);


R13 : gains Ligne (1, 3, 2, 4) -> (3, 1, 2, 4) ; colonne 0


meilleure réponse : 1 -> 0


## 8. Synthèse : la Loi III sur un univers fini

Ce que l'univers fini a permis de trancher, ce que le slogan seul ne pouvait pas :

1. **La séparation est nette et massive.** Sur 3456 paires (jeu, générateur), 2592 flèches préservent la signature de Nash et 864 la transforment. Aucun générateur n'est morphisme pour tout l'espace : l'espèce d'une flèche est une propriété **de la paire**, jamais du générateur seul ni du jeu seul.
2. **La cause couple les deux tables.** Un swap adjacent $R_{(a,b)}$ transforme $G$ si et seulement si une colonne des gains de Ligne contient $\{a,b\}$ **et** cette colonne est une colonne de meilleure réponse de Colonne. La conjecture naïve (condition sur le seul joueur visé) est **fausse** — 288 paires de murs inhabités la réfutent, et l'argument réfuté puis corrigé est resté dans le notebook à dessein.
3. **La géométrie et la grammaire coïncident.** Les 100 points fixes de la grammaire (jeux préservés par les six générateurs) et les 16 jeux les plus exposés (quatre générateurs transformateurs) dessinent la topologie de l'espace ; les murs se traversent partout, mais seuls les **murs habités** changent la chambre Nash. La lecture Bruns-Kimmich (grain D2) devient computationnelle.
4. **Prudence triviale.** Un jeu sans équilibre pur n’est préservé trivialement que si le jeu transformé n’en acquiert pas — la Loi III est substantielle sur les signatures **non vides**, et l'exercice 2 demande de quantifier cette réserve.

La strate 7 du voyage ICT (« certaines transformations changent le monde descriptif lui-même ») gagne ici son premier terrain calibré : changer les préférences n'est pas changer d'avis, et l'écart entre les deux se **mesure** — un quart de l'espace par pas de grammaire, seulement quand le mur traversé est habité.

## 9. Résumé

- **Deux espèces de flèches** coexistent sur l'espace des 576 jeux 2x2 stricts : les swaps de rangs (Robinson-Goforth), qui transforment les préférences, et les morphismes de jeux (Tohmé-Viglizzo), qui transportent les équilibres. Un swap n'est morphisme que **par paire**.
- **Définition finie** : $g$ est morphisme pour $G$ si $\text{Nash}(G) = \text{Nash}(g(G))$ (identité sur les profils). Décidable par énumération : 3456 paires.
- **Comptes mesurés** : chaque générateur adjacent préserve 432/576 jeux (75 %) et en transforme 144 ; 100 jeux sont des points fixes de la grammaire, 16 sont exposés à quatre générateurs.
- **Théorème établi pour un swap adjacent** : $R_{(a,b)}$ transforme $G$ ssi une colonne des gains de Ligne contient $\{a, b\}$ **et** est meilleure réponse de Colonne dans au moins une ligne (symétrique pour $C$). Vérifié sur les 3456 paires ; le compte 432 s'en **dérive** ($16 \times 24$ sans mur $+ 8 \times 6$ au mur inhabité).
- **Contre-exemples exposés** : Stag Hunt et Chicken perdent **strictement** un équilibre (destruction sans compensation) ; le Dilemme ne fait que **déplacer** le sien ; son $R_{34}$ traverse un mur inhabité sans rien changer.
- **Lien aval** : la lecture chambres/murs (grain D2, #12213) est la même structure vue géométriquement ; le chemin minimal certifié (grain D3, #12205 en cible Lean) circulera dans ce graphe dont on sait maintenant quels arcs changent la chambre Nash.

**A retenir :** *la question « transformation ou morphisme ? » n'a de sens que flèche par flèche — et sur un univers fini, chaque flèche se tranche. Le mur ne compte que s'il est habité.*

## À retenir

| Leçon | Mesure dans ce carnet |
|---|---|
| L'espace fini est un terrain d'épreuve, pas une image | `Jeux stricts generes : 576` sur (24 x 24) ; le Dilemme a `1 equilibre(s) : (R1,C1)`, le Stag Hunt `2 equilibre(s) : (R0,C0), (R1,C1)`, Matching Pennies `0 equilibre(s) : aucun` |
| Deux espèces de flèches coexistent sur le même espace | sur les six classiques, `Deadlock` suivi de `R12` est `PRESERVE`, tandis que `Prisoner's Dilemma` suivi de `R12` est `TRANSFORME` |
| Une transformation peut déplacer ou détruire un équilibre | `DEPLACEMENT : (R1,C1) -> (R0,C1)` pour `R12` au Dilemme, mais `Stag Hunt + R12 : perd (R1,C1) sans compensation` |
| La séparation entre les deux espèces est massive | `R12 : 432 preserves / 144 transformes  (75.0 %)` pour chacun des six générateurs, soit `TOTAL : 2592 paires preservantes sur 3456 (75.0 %)` |
| Les jeux que rien ne bouge sont minoritaires | `0 generateur(s) transformateur(s) : 100 jeux`, `1 generateur(s) transformateur(s) : 200 jeux`, `2 generateur(s) transformateur(s) : 180 jeux` ; les `Points fixes (6/6 preserves) : 100 jeux` sont ceux que les six générateurs laissent intacts |
| La cause se prouve, elle ne se devine pas | `Conjecture naive testee sur 3456 paires : 288 desaccords` pour la conjecture naïve, puis `Condition verifiee sur 3456 paires : accords = 3456, desaccords = 0` pour la condition ; la dérivation compte `16 permutations sans paire (sur 24)` et conclut `= 384 + 48 = 432 jeux preserves attendus`, ce que la mesure confirme (`mesure section 4 : 432`) |

**Ce que ce carnet ajoute à la série** : il ne compare pas deux théories sur le papier, il les **confronte sur 576 jeux** et fait trancher la question par un compteur. Le slogan « toute transformation est-elle un morphisme ? » reçoit une réponse chiffrée — 75.0 % des paires préservent l'équilibre — puis la question suivante, plus fine, reçoit une **démonstration** : la condition qui décide de la préservation est énoncée, testée sur 3456 paires, et vérifiée sans exception.

## Vérifiez votre compréhension

**1. Sur les six jeux classiques, `Deadlock` suivi de `R12` est `PRESERVE`, et `Prisoner's Dilemma` suivi de `R12` est `TRANSFORME`. Qu'est-ce qui distingue les deux cas ?**

<details><summary>Réponse</summary>

La flèche est **la même** dans les deux cas : `R12` échange les rangs 1 et 2 de la ligne. Ce qui change, c'est le jeu. Au Deadlock, l'équilibre unique `(R1,C1)` est stable par cet échange — le jeu et sa version transformée ont le même équilibre. Au Dilemme, le même échange déplace l'équilibre : `DEPLACEMENT : (R1,C1) -> (R0,C1)`. Le générateur n'est donc ni « correct » ni « faux » en soi : il est correct pour certains jeux et faux pour d'autres. C'est précisément ce qui définit les **deux espèces de flèches sur un même espace** — et pourquoi l'univers fini, à 576 jeux, était nécessaire pour le montrer.

</details>

**2. L'énumération donne `TOTAL : 2592 paires preservantes sur 3456 (75.0 %)` et `0 generateur(s) transformateur(s) : 100 jeux`. Pourquoi 100 jeux résistent-ils à *tous* les générateurs ?**

<details><summary>Réponse</summary>

Parce que ce ne sont pas des jeux « presque préservés » : ce sont des **points fixes**, et la sortie le dit — `Points fixes (6/6 preserves) : 100 jeux`. Leur structure de Nash est invariante par chacun des six swaps appliqué au jeu initial. Cela ne garantit pas l’invariance par composition : après un premier swap, le jeu intermédiaire peut perdre cette propriété. L'énumération les isole du reste (d'où les 100 jeux à 0 générateur transformateur, contre 200 à un seul), et la caractérisation de la section 6 fournit le versant théorique : une condition testée d'abord comme conjecture (`Conjecture naive testee sur 3456 paires : 288 desaccords`), puis vérifiée (`Condition verifiee sur 3456 paires : accords = 3456, desaccords = 0`), et enfin **dérivée** — `16 permutations sans paire (sur 24)`, `= 384 + 48 = 432 jeux preserves attendus`, ce que la mesure confirme par `mesure section 4 : 432`. La conjecture fausse avait raison sur la forme ; c'est la dérivation qui la remplace.

</details>

## Pour aller plus loin

- [Index de la série GameTheory](README.md) : la place de ce carnet dans l'arc des structures de Nash et des transformations de jeux.
- [Index de la sous-série Coalitions](Coalitions/README.md) : l'autre versant de la théorie des jeux — le pouvoir coalitionnel mesuré sur des jeux de vote pondéré.
- Les trois exercices du carnet (contre-exemple de destruction d'équilibre, cartographie par structure de Nash, swaps non adjacents) reposent tous sur le même protocole : énumérer les 576 jeux stricts et comparer les verdicts générateur par générateur.

### Sources, attribution et dettes de vérification

- **Robinson, D. & Goforth, D.** (2005), *The Topology of the 2×2 Games* — l'espace des jeux ordinaux stricts et ses swaps élémentaires. La représentation informatique et les 576 jeux viennent de [`GameTheory-03-Topology2x2-Python.ipynb`](GameTheory-03-Topology2x2-Python.ipynb).
- **Tohmé, F. & Viglizzo, I.** — catégories concrètes de jeux (`Gam_I`, `Gam`) pour l'espèce préservante. Cadrage cité depuis le chantier 4 (#12207) ; le présent notebook **réduit** la question morphique au test d'égalité des ensembles de Nash sous l'identité sur les profils — une instanciation, pas la théorie catégorique générale.
- **Bruns, B.** — transmutations et lecture chambres/murs, développées dans le grain D2 (#12213).
- **Dettes de vérification héritées du chantier 4** (statut RAPPORTÉ, non établi ici) : le quotient 576 → 144 et le tore à 37 trous sont cités par la source sans dérivation ; ce notebook travaille sur les **576 jeux complets** (sur ensemble du quotient, donc sans le supposer) et le compte par générateur en est indépendant. Les trois références arXiv du chantier n'ont pas été ouvertes firsthand.

See #12237 · See #12207 (chantier 4) · See #12213 (grain D2, chambres et murs)